In [176]:
import numpy as np
import pandas as pd 
import re
from tqdm import tqdm

# Data Re-processing for URL Extraction

## Reading Datasets

In [330]:
BAITBLOCK_DS_PATH = "/kaggle/input/baitblock-dataset"
WRITE_PATH = "/kaggle/working"
DS_PATH = {
    "BERT": BAITBLOCK_DS_PATH + "/BertData.csv",
    "SMS_PHISHING": BAITBLOCK_DS_PATH + "/SmsPhishingDataset.csv",
    "SMS_SPAM": WRITE_PATH + "/SmsSpam_utf8.csv",
    "CEAS_08": BAITBLOCK_DS_PATH + "/CEAS-08.csv",
    "TREC_07": BAITBLOCK_DS_PATH + "/TREC-07.csv",
}

In [322]:
def readCSV(file_path):
    return pd.read_csv(file_path)

def saveCSV(df, file_path):
    df.to_csv(file_path, index=False)
    print(f"DataFrame successfully saved to {file_path}")

In [ ]:
def from_encoding_to_utf8(pathFrom,pathTo,encodingFrom):
    with open(pathFrom, 'r', encoding=encodingFrom) as file:
        content = file.read()
    with open(pathTo, 'w', encoding='utf-8') as file:
        file.write(content)
    print(f"File has been converted from {encodingFrom} to utf8 and saved as {pathTo}.")

In [323]:
from_encoding_to_utf8(DS_PATH["SMS_SPAM"],WRITE_PATH + "/SmsSpam_utf8.csv",'Windows-1252')

File has been converted from Windows-1252 to utf8 and saved as /kaggle/working/SmsSpam_utf8.csv.


In [331]:
bert_df = readCSV(DS_PATH["BERT"])
sms_phish_df = readCSV(DS_PATH["SMS_PHISHING"])
sms_spam_df = readCSV(DS_PATH["SMS_SPAM"])
ceas_08_df = readCSV(DS_PATH["CEAS_08"])
trec_07_df = readCSV(DS_PATH["TREC_07"])

## Unneeded Columns Removal

In [ ]:
def removeColsByIndex(df, cols_to_remove):
    df = df.drop(df.columns[cols_to_remove], axis=1)
    return df

In [338]:
sms_phish_df = removeColsByIndex(sms_phish_df,[0])
print(sms_phish_df)
saveCSV(sms_phish_df,WRITE_PATH + "/SmsPhish_slct.csv")

                                                   text  url
0     Your opinion about me? 1. Over 2. Jada 3. Kusr...    0
1     What's up? Do you want me to come online? If y...    0
2                          So u workin overtime nigpun?    0
3     Also sir, i sent you an email about how to log...    0
4     Please Stay At Home. To encourage the notion o...    0
...                                                 ...  ...
5966                           :( but your not here....    0
5967  Becoz its  &lt;#&gt;  jan whn al the post ofic...    0
5968  Its a valentine game. . . send dis msg to all ...    0
5969                              We r outside already.    0
5970  The Xmas story is peace.. The Xmas msg is love...    0

[5971 rows x 2 columns]
DataFrame successfully saved to /kaggle/working/SmsPhish_slct.csv


In [339]:
sms_spam_df = removeColsByIndex(sms_spam_df,[0,2,3,4])
print(sms_spam_df)
saveCSV(sms_spam_df,WRITE_PATH + "/SmsSpam_utf8_slct.csv")

                                                   text
0     Go until jurong point, crazy.. Available only ...
1                         Ok lar... Joking wif u oni...
2     Free entry in 2 a wkly comp to win FA Cup fina...
3     U dun say so early hor... U c already then say...
4     Nah I don't think he goes to usf, he lives aro...
...                                                 ...
5567  This is the 2nd time we have tried 2 contact u...
5568              Will Ì_ b going to esplanade fr home?
5569  Pity, * was in mood for that. So...any other s...
5570  The guy did some bitching but I acted like i'd...
5571                         Rofl. Its true to its name

[5572 rows x 1 columns]
DataFrame successfully saved to /kaggle/working/SmsSpam_utf8_slct.csv


In [340]:
ceas_08_df = removeColsByIndex(ceas_08_df,[0,1,2,5])
print(ceas_08_df)
saveCSV(ceas_08_df,WRITE_PATH + "/ceas_08_slct.csv")

                                                 subject  \
0                              Never agree to be a loser   
1                                 Befriend Jenna Jameson   
2                                   CNN.com Daily Top 10   
3      Re: svn commit: r619753 - in /spamassassin/tru...   
4                             SpecialPricesPharmMoreinfo   
...                                                  ...   
39149                        CNN Alerts: My Custom Alert   
39150                        CNN Alerts: My Custom Alert   
39151                                   Slideshow viewer   
39152                              Note on 2-digit years   
39153                      [Python-Dev] PEP 370 heads up   

                                                    body  urls  
0      Buck up, your troubles caused by small dimensi...     1  
1      \nUpgrade your sex and pleasures with these te...     1  
2      >+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...     1  
3      Would anyone

In [341]:
trec_07_df = removeColsByIndex(trec_07_df,[0,1,2,5])
print(trec_07_df)
saveCSV(trec_07_df,WRITE_PATH + "/trec_07_df_slct.csv")

                                                 subject  \
0                      Generic Cialis, branded quality@    
1                                 Typo in /debian/README   
2                                       authentic viagra   
3                                   Nice talking with ya   
4      or trembling; stomach cramps; trouble in sleep...   
...                                                  ...   
53752                                 Job: just for you.   
53753  the reply for your request for a job place [le...   
53754  Re: [R] Me again, about the horrible documenta...   
53755                              Re: [R] RODBC problem   
53756  I wanted the desk at his own laws: of the.  Bu...   

                                                    body  urls  
0      \n\n\n\n\n\n\nDo you feel the pressure to perf...     0  
1      Hi, i've just updated from the gulus and I che...     1  
2      Mega  authenticV I A G R A   $ DISCOUNT priceC...     1  
3      \nHey Billy,

In [344]:
sms_phish_df = readCSV(WRITE_PATH + "/SmsPhish_slct.csv")
sms_spam_df = readCSV(WRITE_PATH + "/SmsSpam_utf8_slct.csv")
ceas_08_df = readCSV(WRITE_PATH + "/ceas_08_slct.csv")
trec_07_df = readCSV(WRITE_PATH + "/trec_07_df_slct.csv")

## Removing Over Spacing

In [53]:
def eliminateOverSpacing(df,col):
    df[col] = df[col].str.replace('\s+', ' ', regex=True).str.strip()
    return df

In [345]:
sms_spam_df = eliminateOverSpacing(sms_spam_df,'text')
sms_phish_df = eliminateOverSpacing(sms_phish_df,'text')
ceas_08_df = eliminateOverSpacing(ceas_08_df,'body')
ceas_08_df = eliminateOverSpacing(ceas_08_df,'subject')
trec_07_df = eliminateOverSpacing(trec_07_df,'body')
trec_07_df = eliminateOverSpacing(trec_07_df,'subject')

In [346]:
saveCSV(sms_phish_df,WRITE_PATH + "/SmsPhish_nospace.csv")
saveCSV(sms_spam_df,WRITE_PATH + "/SmsSpam_nospace.csv")
saveCSV(ceas_08_df,WRITE_PATH + "/ceas_08_nospace.csv")
saveCSV(trec_07_df,WRITE_PATH + "/trec_07_nospace.csv")

DataFrame successfully saved to /kaggle/working/SmsPhish_nospace.csv
DataFrame successfully saved to /kaggle/working/SmsSpam_nospace.csv
DataFrame successfully saved to /kaggle/working/ceas_08_nospace.csv
DataFrame successfully saved to /kaggle/working/trec_07_nospace.csv


In [284]:
sms_phish_df = readCSV(WRITE_PATH + "/SmsPhish_nospace.csv")
sms_spam_df = readCSV(WRITE_PATH + "/SmsSpam_nospace.csv")
ceas_08_df = readCSV(WRITE_PATH + "/ceas_08_nospace.csv")
trec_07_df = readCSV(WRITE_PATH + "/trec_07_nospace.csv")

## Extracting URLs

In [347]:
def extractURLUnsupervised(df, textCol):
    url_pattern = r'http\S+|www\S+|https\S+'

    def check_url(text):
        match = re.search(url_pattern, text)
        if match:
            url = match.group(0)
            cleaned_text = re.sub(url_pattern, '', text, flags=re.MULTILINE)
            return 1, url, cleaned_text
        else:
            return 0, None, text
    
    df[['isURL', 'url', textCol]] = df[textCol].apply(lambda x: check_url(str(x))).apply(pd.Series)
    df['isURL'] = df['isURL'].astype(int)
    return df

In [264]:
print(re.search(r'(http://\S+|www\S+|https://\S+)', "BugFix10287 SAVE AS, We offer a selection of the highest quality replica watches available today Genuine watches of these replicas are very expensive and the choice of the rich famous and collectors around the world These watches were designed with the greatest detail and craftsmanship Replica Watches are inexpensive and sometimes give the impression that you are wearing the Genuine while you are catering the posh cocktail party www.sedecce.com A DAT drive is used to back up large amounts of information such as all the files on a network A DAT cartridge can store up to 8 Gigabytes of information the equivalent of 10 or more CDROM discs See also QIC DRIVE A window you can open to adjust various parts of your computer You can use the Control Panel to adjust how fast your mouse moves the colors on your screen the volume of your speakers the time and date on your computer and so on Naming a cell in a spreadsheet with dollar signs example A14 to point to a fixed number you often use like a sales tax rate An absolute reference is locked in so it doesnt change even if you copy your formula to a different cell See also RELATIVE REFERENCE A key you can use to give you another set of commands Ctrl commands are commonly used shortcuts For example pressing CtrlS in many programs saves your document faster than selecting Save from the File menu To change the view of your files so you see your main directories folders as well as any files inside them Also to change the view of a document or presentation so you see both your main headings and underlying text To throw away a file you no longer need If youre lucky you can still recover a file that you deleted by mistake See also RECOVER An early version of a software product thats not quite ready for sale Beta software is given to carefully selected users who try it out and report back any problems or suggested improvements"))

None


In [348]:
def extractURLSupervised(df, textCol, isURLCol):
    url_pattern = r'http\S+|www\S+|https\S+'
    
    def extract_url_if_needed(row):
        if "Yours Sincerely Quincy" in row[textCol]:
            print(row[textCol])
            print(re.search(url_pattern, row[textCol]))
        if row[isURLCol] == 1:
            match = re.search(url_pattern, row[textCol])
            if match:
                url = match.group(0)
                cleaned_text = re.sub(url_pattern, '', row[textCol], flags=re.MULTILINE)
                return 1, url, cleaned_text
        return 0, None, row[textCol]
    
    df[[isURLCol,'url', textCol]] = df.apply(extract_url_if_needed, axis=1).apply(pd.Series)
    return df

In [349]:
sms_spam_df = extractURLUnsupervised(sms_spam_df,'text')
sms_phish_df = sms_phish_df.rename(columns={'url': 'isURL'})
sms_phish_df = extractURLSupervised(sms_phish_df,'text','isURL')
ceas_08_df = ceas_08_df.rename(columns={'urls': 'isURL'})
ceas_08_df = extractURLSupervised(ceas_08_df,'body','isURL')
trec_07_df = trec_07_df.rename(columns={'urls': 'isURL'})
trec_07_df = extractURLSupervised(trec_07_df,'body','isURL')

In [297]:
def removeURLs(df,col):
    df[col] = df[col].apply(lambda x: re.sub(r'http\S+|www\S+|https\S+', '', str(x), flags=re.MULTILINE))
    return df

In [350]:
sms_spam_df = removeURLs(sms_spam_df,'text')
sms_phish_df = removeURLs(sms_phish_df,'text')
ceas_08_df = removeURLs(ceas_08_df,'body')
trec_07_df = removeURLs(trec_07_df,'body')

In [351]:
saveCSV(sms_phish_df,WRITE_PATH + "/SmsPhish_nospace_url.csv")
saveCSV(sms_spam_df,WRITE_PATH + "/SmsSpam_nospace_url.csv")
saveCSV(ceas_08_df,WRITE_PATH + "/ceas_08_nospace_url.csv")
saveCSV(trec_07_df,WRITE_PATH + "/trec_07_nospace_url.csv")

DataFrame successfully saved to /kaggle/working/SmsPhish_nospace_url.csv
DataFrame successfully saved to /kaggle/working/SmsSpam_nospace_url.csv
DataFrame successfully saved to /kaggle/working/ceas_08_nospace_url.csv
DataFrame successfully saved to /kaggle/working/trec_07_nospace_url.csv


## Removing Special Characters

In [213]:
def removeSC(df,col):
    df[col] = df[col].apply(lambda x: re.sub(r'[^A-Za-z0-9\s]', '', str(x)))
    return df

In [352]:
sms_spam_df = removeSC(sms_spam_df,'text')
sms_phish_df = removeSC(sms_phish_df,'text')
ceas_08_df = removeSC(ceas_08_df,'body')
ceas_08_df = removeSC(ceas_08_df,'subject')
trec_07_df = removeSC(trec_07_df,'body')
trec_07_df = removeSC(trec_07_df,'subject')

In [353]:
saveCSV(sms_phish_df,WRITE_PATH + "/SmsPhish_nospace_url_sc.csv")
saveCSV(sms_spam_df,WRITE_PATH + "/SmsSpam_nospace_url_sc.csv")
saveCSV(ceas_08_df,WRITE_PATH + "/ceas_08_nospace_url_sc.csv")
saveCSV(trec_07_df,WRITE_PATH + "/trec_07_nospace_url_sc.csv")

DataFrame successfully saved to /kaggle/working/SmsPhish_nospace_url_sc.csv
DataFrame successfully saved to /kaggle/working/SmsSpam_nospace_url_sc.csv
DataFrame successfully saved to /kaggle/working/ceas_08_nospace_url_sc.csv
DataFrame successfully saved to /kaggle/working/trec_07_nospace_url_sc.csv


## Joining Columns

In [137]:
def joinCols(df,col1,col2,newCol):
    df[newCol] = df[col2].fillna('') + ', ' + df[col1].fillna('')
    df = df.drop(columns=[col1,col2])
    return df

In [354]:
ceas_08_df = joinCols(ceas_08_df,'body','subject','text')
trec_07_df = joinCols(trec_07_df,'body','subject','text')
ceas_08_df = ceas_08_df[['text','isURL','url']]
trec_07_df = trec_07_df[['text','isURL','url']]

In [355]:
saveCSV(ceas_08_df,WRITE_PATH + "/ceas_08_nospace_url_sc_joined.csv")
saveCSV(trec_07_df,WRITE_PATH + "/trec_07_nospace_url_sc_joined.csv")

DataFrame successfully saved to /kaggle/working/ceas_08_nospace_url_sc_joined.csv
DataFrame successfully saved to /kaggle/working/trec_07_nospace_url_sc_joined.csv


## Grouping into One Dataset

In [356]:
bert_url = pd.concat([sms_spam_df,sms_phish_df,ceas_08_df,trec_07_df], ignore_index=True)

In [357]:
saveCSV(bert_url,WRITE_PATH + "/bert_url.csv")

DataFrame successfully saved to /kaggle/working/bert_url.csv


## Re-eliminating Spaces

In [358]:
bert_url = eliminateOverSpacing(bert_url,'text')

In [359]:
saveCSV(bert_url,WRITE_PATH + "/bert_url_nospaces.csv")

DataFrame successfully saved to /kaggle/working/bert_url_nospaces.csv


In [360]:
bert_df = eliminateOverSpacing(bert_df,'text')
saveCSV(bert_url,WRITE_PATH + "/bert_df_nospaces.csv")

DataFrame successfully saved to /kaggle/working/bert_df_nospaces.csv


In [226]:
bert_url = readCSV(WRITE_PATH + "/bert_url_nospaces.csv")

## Adding URLs to Model BERT Data

In [361]:
def update_target_from_source(target_df, source_df, column, isURL_col, url_col, not_found_path):
    source_df = source_df.drop_duplicates(subset=[column])
    source_dict = source_df.set_index(column)[[isURL_col, url_col]].to_dict(orient='index')
    not_found_df = pd.DataFrame(columns=target_df.columns)
    count = 0
    success_count = 0
    def update_row(row):
        nonlocal count
        count += 1
        print(f"{count}/{target_df.shape[0]}",end="")
        value = row[column]
        if value in source_dict:
            nonlocal success_count
            success_count += 1
            print(f" | {success_count}")
            return pd.Series({
                isURL_col: source_dict[value][isURL_col],
                url_col: source_dict[value][url_col]
            })
        else:
            print()
            nonlocal not_found_df
            not_found_df = pd.concat([not_found_df, pd.DataFrame([row])], ignore_index=True)
            return pd.Series({
                isURL_col: 0,
                url_col: None
            })
    target_df[[isURL_col, url_col]] = target_df.apply(update_row, axis=1)
    target_df['isURL'] = target_df['isURL'].astype(int)
    saveCSV(not_found_df,not_found_path)
    return target_df

In [ ]:
bert_df = update_target_from_source(bert_df,bert_url,'text','isURL','url',WRITE_PATH + "/not_found_3.csv")

In [363]:
saveCSV(bert_df,'BaitBlockJointDataset.csv')

DataFrame successfully saved to BaitBlockJointDataset.csv
